# Notebook 1: Data Gathering

**Car Price Prediction Pipeline**

---

This notebook loads the raw car listings dataset into the Databricks Catalog

**Input:** `workspace.default.cars`

**Output:** `workspace.default.cars_new`

## Load CSV with Pandas

In [0]:
import pandas as pd

# 1. Gather data from the table
df = spark.table("workspace.default.cars").toPandas()


print(f"Rows: {len(df)}")
print(f"Columns: {len(df.columns)}")

Rows: 56244
Columns: 12


## Basic Inspection

In [0]:
# Data types
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56244 entries, 0 to 56243
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   make                 56244 non-null  object 
 1   model                56244 non-null  object 
 2   priceUSD             56244 non-null  int64  
 3   year                 56244 non-null  int64  
 4   condition            56244 non-null  object 
 5   mileage(kilometers)  56244 non-null  float64
 6   fuel_type            56244 non-null  object 
 7   volume(cm3)          56244 non-null  object 
 8   color                56244 non-null  object 
 9   transmission         56244 non-null  object 
 10  drive_unit           54339 non-null  object 
 11  segment              50953 non-null  object 
dtypes: float64(1), int64(2), object(9)
memory usage: 5.1+ MB


volume(cm3) Dtype should be changed to numeric

In [0]:
# Preview first rows
df.head(10)

,make,model,priceUSD,year,condition,mileage(kilometers),fuel_type,volume(cm3),color,transmission,drive_unit,segment
0,mazda,2,5500,2008,with mileage,162000.0,petrol,1500,burgundy,mechanics,front-wheel drive,B
1,mazda,2,5350,2009,with mileage,120000.0,petrol,1300,black,mechanics,front-wheel drive,B
2,mazda,2,7000,2009,with mileage,61000.0,petrol,1500,silver,auto,front-wheel drive,B
3,mazda,2,3300,2003,with mileage,265000.0,diesel,1400,white,mechanics,front-wheel drive,B
4,mazda,2,5200,2008,with mileage,97183.0,diesel,1400,gray,mechanics,front-wheel drive,B
5,mazda,2,3400,2005,with mileage,150000.0,petrol,1300,blue,mechanics,front-wheel drive,B
6,mazda,2,5000,2008,with mileage,112000.0,petrol,1300,gray,mechanics,front-wheel drive,B
7,mazda,2,7300,2009,with mileage,95000.0,petrol,1500,other,auto,front-wheel drive,B
8,mazda,2,6400,2010,with mileage,93804.0,petrol,1500,other,mechanics,front-wheel drive,B
9,mazda,2,6132,2008,with mileage,196511.0,petrol,1500,other,auto,front-wheel drive,B


In [0]:
# Descriptive statistics for numeric columns
df.describe().round(2)

,priceUSD,year,mileage(kilometers)
count,56244.00,56244.00,56244.00
mean,7415.46,2003.45,244395.63
std,8316.96,8.14,321030.67
min,48.00,1910.00,0.00
25%,2350.00,1998.00,137000.00
50%,5350.00,2004.00,228500.00
75%,9807.50,2010.00,310000.00
max,235235.00,2019.00,9999999.00


In [0]:
# Value counts for categorical columns
categorical_cols = ['make', 'condition', 'fuel_type', 'color', 'transmission', 'drive_unit', 'segment']

for col_name in categorical_cols:
    print(f"\n--- {col_name} ---")
    print(df[col_name].value_counts().head(15).to_string())


--- make ---
volkswagen       6861
audi             4030
bmw              4013
opel             3779
renault          3713
mercedes-benz    3541
ford             3078
peugeot          2876
nissan           2233
toyota           2177
mazda            2006
citroen          1991
hyundai          1499
mitsubishi       1350
volvo            1232

--- condition ---
with mileage    55278
with damage       512
for parts         454

--- fuel_type ---
petrol        36405
diesel        19792
electrocar       47

--- color ---
black       12385
silver      10075
blue         8083
gray         5807
white        5292
green        3911
other        3397
red          2744
burgundy     2026
brown        1349
purple        625
yellow        317
orange        233

--- transmission ---
mechanics    36056
auto         20188

--- drive_unit ---
front-wheel drive             38016
rear drive                     6836
all-wheel drive                5890
part-time four-wheel drive     3597

--- segment ---
D 

In [0]:
# Unique counts
print("Unique value counts:")
print("=" * 40)
for col_name in df.columns:
    n_unique = df[col_name].nunique()
    print(f"  {col_name:25s}: {n_unique}")

Unique value counts:
  make                     : 96
  model                    : 1034
  priceUSD                 : 2970
  year                     : 78
  condition                : 3
  mileage(kilometers)      : 8400
  fuel_type                : 3
  volume(cm3)              : 459
  color                    : 13
  transmission             : 2
  drive_unit               : 4
  segment                  : 9


In [0]:
print(df['segment'].unique())

['B' 'C' None 'M' 'D' 'E' 'A' 'J' 'S' 'F']


In [0]:
print(df['volume(cm3)'].unique())

['1500' '1300' '1400' '1600' '2000' '2300' '20000' '2500' '1999' '1560'
 '1598' '1998' '1800' '3000' '2200' '2498' '2400' '2330' '1790' '1900'
 '1700' '1721' '1390' '1794' '1200' '1000' '2445' '2432' '2440' '2100'
 '16000' '2450' '2900' '3200' '3300' '2700' '1595' '1781' '1984' '1798'
 '2600' '2800' '1780' '1750' '1799' '1987' '500' '1599' '3800' '4200'
 '2333' '2370' '2226' '2309' '1921' '2196' '1989' '2193' '2460' '1789'
 '2790' '1760' '1926' '18000' '1100' '1910' 'nan' '2487' '14000' '1340'
 '1360' '1868' '1450' '1444' '1198' '1199' '1888' '1396' '1761' '1587'
 '13000' '1324' '1480' '1220' '19000' '1593' '2230' '2088' '1650' '1891'
 '1358' '1609' '888' '1394' '900' '1995' '2111' '2222' '1840' '1991'
 '1796' '1997' '2179' '3600' '700' '800' '1197' '1397' '1234' '1119'
 '1399' '12000' '1111' '1333' '1294' '1290' '1596' '10000' '5555' '1666'
 '1569' '1458' '1499' '1585' '1490' '1380' '1445' '2286' '4700' '2693'
 '15000' '1509' '5000' '2996' '1597' '5500' '1996' '1496' '1690' '1895'
 '1

## 4. Save to Catalog

In [0]:
# Convert pandas DataFrame to Spark and save as Delta table in Catalog

df_renamed = df.rename(columns={
    'mileage(kilometers)': 'mileage_kilometers',
    'volume(cm3)': 'volume_cm3'
})
df_spark = spark.createDataFrame(df_renamed)
df_spark.write.mode("overwrite").saveAsTable("workspace.default.cars_new")

# Verify
verify = spark.table("workspace.default.cars_new").toPandas()
print(f"Saved raw_data: ({len(verify)} rows, {len(verify.columns)} columns)")
print(f"Table location: workspace.default.cars_new")

Saved raw_data: (56244 rows, 12 columns)
Table location: workspace.default.cars_new
